# LoRA reproduction: RoBERTa-base / SST-2
Public workflow candidate. Select a free GPU runtime before running.
This notebook trains independent repetitions; it does not replace frozen Stage 5 results.
Full GPU execution of this public packaging has not yet been validated.
No private Drive account is required. The first code cell clones the public repository when absent.


In [ ]:
from pathlib import Path
import sys, subprocess, os
OUTPUT_PARENT = Path('/content') if Path('/content').exists() else Path('/kaggle/working')
REPO = OUTPUT_PARENT / 'CMP4509-LoRA-Reproduction'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ykarakoca/CMP4509-LoRA-Reproduction.git', str(REPO)], check=True)
assert (REPO / 'scripts/reproduce.py').is_file(), 'Repository files not found.'


## Install dependencies
Keep the platform PyTorch/CUDA installation. If these recorded package pins are unavailable, stop and report the error; do not silently substitute versions. Restart and rerun if the platform requests it.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt')], check=True)


## Check GPU and record environment
Strict mode additionally requires the exact Stage 5 package versions and Tesla T4. Portable mode records actual versions and hardware; numerical identity is not guaranteed.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime.'
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))


## Audit the frozen evidence
CPU analysis only; this is not training or checkpoint restoration.

In [ ]:
AUDIT = OUTPUT_PARENT / 'lora-frozen-audit'
subprocess.run([sys.executable, str(REPO / 'scripts/analyze.py'), '--output', str(AUDIT)], check=True)


## Train and evaluate
Default: all four methods × three seeds, five epochs. MODE="smoke" runs four 64-example checks and never evaluates the final set. Use separate output directories for modes. Completed full runs can be skipped after checks; interrupted training must restart that run, because optimizer/RNG states are not stored. All 12 checkpoints are fixed before final evaluation. Save/download the output directory before the runtime expires.

In [ ]:
MODE = 'full'
STRICT_ENV = False
RUNS = OUTPUT_PARENT / ('lora-public-' + MODE)
command = [sys.executable, '-u', str(REPO / 'scripts/reproduce.py'), '--mode', MODE, '--output', str(RUNS)]
if STRICT_ENV: command.append('--strict-env')
subprocess.run(command, check=True)


## Inspect newly generated results
Smoke outputs are engineering checks only. Full results and figures come from the new checkpoints.

In [ ]:
from IPython.display import display, Image
if MODE == 'full':
    print((RUNS / 'analysis/summary.csv').read_text())
    display(Image(filename=str(RUNS / 'analysis/accuracy_efficiency.png')))
    display(Image(filename=str(RUNS / 'analysis/tuning_curves.png')))
else:
    print('Smoke complete; no final scientific metrics generated.')
